# U.S. Electricity Generation Forecasting — LSTM

This notebook is part of a collaborative DAEN 430 final project comparing a seasonal ARIMA model with an LSTM for monthly U.S. electricity generation forecasting.

**Authors:** Maddie Bird, Henry Supp, Jade Winebright  
**Data:** 142 monthly observations, January 1985–October 1996  
**Approach:** chronological 80/20 train/test split, 12-month sliding window, stacked LSTM with dropout/L2 regularization and early stopping.



## Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.stattools import acf

## Load data

In [ ]:
path = "data/electricity.csv"


In [ ]:
df = pd.read_csv(path)
series = df.select_dtypes(include=[np.number]).values.flatten().astype(float)

In [ ]:
start_year = 1985
freq = 12  # monthly

n = len(series)
time_index = np.arange(n) / freq + start_year

In [ ]:
n

## Chronological train/test split

In [ ]:
train_size = int(np.floor(0.8 * n))

train = series[:train_size]
test  = series[train_size:]

In [ ]:
train.size

## Scale data and create 12-month sequences

In [ ]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train.reshape(-1,1))
test_scaled  = scaler.transform(test.reshape(-1,1))


In [ ]:
def create_sequences(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X), np.array(y)

window = 12

X_train, y_train = create_sequences(train_scaled, window)
combined = np.concatenate((train_scaled[-window:], test_scaled))
X_test, y_test = create_sequences(combined, window)

X_train = X_train.reshape((X_train.shape[0], window, 1))
X_test  = X_test.reshape((X_test.shape[0], window, 1))


## Build and train the LSTM

In [ ]:
model = Sequential([
    Input(shape=(window, 1)),
    LSTM(32, return_sequences=True, activation='tanh',
         kernel_regularizer=l2(1e-4)), Dropout(0.2),

    LSTM(16, activation='tanh',
         kernel_regularizer=l2(1e-4)), Dropout(0.2),

    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
import time

start_time = time.time()

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=8,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

end_time = time.time()
lstm_train_time = end_time - start_time
print(f"LSTM Training Time: {lstm_train_time:.2f} seconds")


## Evaluate predictions

In [ ]:
train_pred = model.predict(X_train).flatten()
test_pred  = model.predict(X_test).flatten()

# Inverse scale
train_pred = scaler.inverse_transform(train_pred.reshape(-1,1)).flatten()
test_pred  = scaler.inverse_transform(test_pred.reshape(-1,1)).flatten()

train_actual = train[window:]
test_actual = test[:len(test_pred)]

In [ ]:
def accuracy_table(train_actual, train_pred, test_actual, test_pred, seasonality=12):

    def compute_metrics(actual, pred):
        errors = actual - pred

        ME = np.mean(errors)
        RMSE = np.sqrt(np.mean(errors**2))
        MAE = np.mean(np.abs(errors))

        nonzero = actual != 0
        MPE = np.mean(errors[nonzero] / actual[nonzero]) * 100
        MAPE = np.mean(np.abs(errors[nonzero] / actual[nonzero])) * 100

        naive = actual[seasonality:] - actual[:-seasonality]
        denom = np.mean(np.abs(naive)) if len(naive) > 0 else np.nan
        MASE = MAE / denom if denom != 0 else np.nan

        try:
            ACF1 = acf(errors, nlags=1)[1]
        except:
            ACF1 = np.nan

        num = np.sqrt(np.mean((pred - actual)**2))
        den = np.sqrt(np.mean(actual[:-1]**2)) + np.sqrt(np.mean(pred[:-1]**2))
        Theils_U = num / den if den != 0 else np.nan

        return [ME, RMSE, MAE, MPE, MAPE, MASE, ACF1, Theils_U]

    return pd.DataFrame(
        [
            compute_metrics(train_actual, train_pred),
            compute_metrics(test_actual, test_pred)
        ],
        index=["Training set", "Test set"],
        columns=["ME", "RMSE", "MAE", "MPE", "MAPE", "MASE", "ACF1", "Theil's U"]
    )

print(accuracy_table(train_actual, train_pred, test_actual, test_pred))



## Forecast vs. actual

In [ ]:
plt.figure(figsize=(9,5))

# Full actual series
plt.plot(time_index, series, label="Actual", linewidth=1, color="black")

# LSTM forecast aligned EXACTLY like R forecast horizon
test_time = time_index[train_size:train_size + len(test_pred)]

plt.plot(test_time, test_pred, label="LSTM Forecast", linewidth=1, color="red")

# Test region shading
plt.axvspan(time_index[train_size], time_index[-1], alpha=0.15, label="Test Period")

plt.title("LSTM Prediction vs Actual")
plt.xlabel("Time")
plt.ylabel("Electricity Generation")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9,5))

# Plot forecast first (so it matches R style layering)
plt.plot(test_time, test_pred, color="red", linewidth=1, label="Forecast")

# Plot full actual series
plt.plot(time_index, series, color="black", linewidth=1, label="Actual")

# Titles and labels (match wording)
plt.title("LSTM Forecast vs Actual")
plt.xlabel("Time")
plt.ylabel("Electricity Generation")

# Legend (top-left, simple)
plt.legend(loc="upper left", frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,6), dpi=200)

plt.plot(test_time, test_pred, color="red", linewidth=1.5, label="Forecast")
plt.plot(time_index, series, color="black", linewidth=1.5, label="Actual")

plt.title("LSTM Forecast vs Actual", fontsize=18)
plt.xlabel("Time", fontsize=14)
plt.ylabel("Electricity Generation", fontsize=14)

plt.legend(loc="upper left", frameon=False, fontsize=12)
plt.tight_layout()
plt.show()
